#**Pandas dataframe to spark Dataframe**

In [0]:
csv_data = """Id,Name,Age,City,Salary
1,Alice,25,Bangalore,65000
2,Bob,34,Hyderabad,82000
3,Charlie,29,Chennai,71000
4,David,41,Pune,98000
5,Eva,23,Delhi,54000
6,Frank,thirty,Mumbai,75000        -- corrupted: Age is text instead of integer
7,Grace,27,,62000                  -- corrupted: missing City
8,Hank,,Kolkata,88000              -- corrupted: missing Age
9,Ivy,31,Jaipur,not_available      -- corrupted: Salary is text instead of integer
10,Jack,28,Bhopal                  -- corrupted: missing Salary
"""

In [0]:
import pandas as pd #for handling tabular data in-memory
from io import StringIO #lets you treat a string like a file — perfect for simulating CSV input without needing an actual file.

#corrupted data
csv_data = """Id,Name,Age,City,Salary
1,Alice,25,Bangalore,65000
2,Bob,34,Hyderabad,82000
3,Charlie,29,Chennai,71000
4,David,41,Pune,98000
5,Eva,23,Delhi,54000
6,Frank,thirty,Mumbai,75000
7,Grace,27,,62000                  
8,Hank,,Kolkata,88000              
9,Ivy,31,Jaipur,not_available      
10,Jack,28,Bhopal                  
"""

pdf = pd.read_csv(StringIO(csv_data), on_bad_lines='skip')
# - **`StringIO(csv_data)`**: Converts the string into a file-like object.
# - **`pd.read_csv(...)`**: Reads the CSV into a Pandas DataFrame. It is a pandas functions not a databricks function
# - **on_bad_lines='skip'**: Tells Pandas to **skip rows** that don’t match the expected number of columns.


pdf.head(10)

,Id,Name,Age,City,Salary
0,1,Alice,25,Bangalore,65000
1,2,Bob,34,Hyderabad,82000
2,3,Charlie,29,Chennai,71000
3,4,David,41,Pune,98000
4,5,Eva,23,Delhi,54000
5,6,Frank,thirty,Mumbai,75000
6,7,Grace,27,NaN,62000
7,8,Hank,NaN,Kolkata,88000
8,9,Ivy,31,Jaipur,not_available
9,10,Jack,28,Bhopal,NaN


In [0]:
df = spark.createDataFrame(pdf) #Converts the Pandas DataFrame (`pdf`) into a **Spark DataFrame (`df`)**.Spark DataFrames are distributed across the cluster unlike pandas dataframe that are stored on a single machine in memor, so they can handle much larger datasets than Pandas.
#so however both may look similar on surface but are different

df.show()
df.printSchema()

+---+-------+------+--------------------+--------------------+
| Id|   Name|   Age|                City|              Salary|
+---+-------+------+--------------------+--------------------+
|  1|  Alice|    25|           Bangalore|               65000|
|  2|    Bob|    34|           Hyderabad|               82000|
|  3|Charlie|    29|             Chennai|               71000|
|  4|  David|    41|                Pune|               98000|
|  5|    Eva|    23|               Delhi|               54000|
|  6|  Frank|thirty|              Mumbai|               75000|
|  7|  Grace|    27|                NULL|62000            ...|
|  8|   Hank|  NULL|             Kolkata| 88000              |
|  9|    Ivy|    31|              Jaipur| not_available      |
| 10|   Jack|    28|Bhopal           ...|                NULL|
+---+-------+------+--------------------+--------------------+

root
 |-- Id: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- City

#Printing Bad Records
##For Printing bad/Corrupted records first we need to define the schema

# _corrupt_record

This approach only works when reading from a file — not from Pandas — because Spark handles parsing and corruption detection during ingestion.

So need to ingest data and create a file Hence created a corrupted csv file at 
### /Volumes/workspace/default/workspace_volume/Corrupted_records.csv

In [0]:
path="/Volumes/workspace/default/workspace_volume/Corrupted_records.csv"

from pyspark.sql.types import StructType, StructField, StringType,IntegerType

manual_schema = StructType([
    StructField("Id", StringType(), True),
    StructField("Person Name", StringType(), True),
    StructField("Person age", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Salary", StringType(), True),
    StructField("_corrupt_record",StringType(), True)
])

In [0]:
df_refine = spark.read.format("csv")\
    .option("header", True)\
    .schema(manual_schema)\
    .option("mode","PERMISSIVE")\
    .csv(path)

df_refine.printSchema()
df_refine.show()

root
 |-- Id: string (nullable = true)
 |-- Person Name: string (nullable = true)
 |-- Person age: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: string (nullable = true)
 |-- _corrupt_record: string (nullable = true)

+---+-----------+----------+--------------------+--------------------+--------------------+
| Id|Person Name|Person age|                City|              Salary|     _corrupt_record|
+---+-----------+----------+--------------------+--------------------+--------------------+
|  1|      Alice|        25|           Bangalore|               65000|                NULL|
|  2|        Bob|        34|           Hyderabad|               82000|                NULL|
|  3|    Charlie|        29|             Chennai|               71000|                NULL|
|  4|      David|        41|                Pune|               98000|                NULL|
|  5|        Eva|        23|               Delhi|               54000|                NULL|
|  6|      Frank| 

In [0]:
df_refine.show(truncate=False) # to show  all records in detail

+---+-----------+----------+----------------------------------------------------+------------------------------------------------------------------+---------------------------------------------------------------+
|Id |Person Name|Person age|City                                                |Salary                                                            |_corrupt_record                                                |
+---+-----------+----------+----------------------------------------------------+------------------------------------------------------------------+---------------------------------------------------------------+
|1  |Alice      |25        |Bangalore                                           |65000                                                             |NULL                                                           |
|2  |Bob        |34        |Hyderabad                                           |82000                                                             |

#storing corrupt records

##.option("badRecordsPath")

In [0]:
df_refine = spark.read.format("csv")\
    .option("header", True)\
    .schema(manual_schema)\
    .option("mode","PERMISSIVE")\
    .option("badRecordsPath","/Volumes/workspace/default/workspace_volume/saveBadRecords/Corrupted_records.csv")\
    .csv(path)

In [0]:
#reading the created file